# Sales Forecast AI - Model Benchmarking and Comparison

This notebook demonstrates the end-to-end training, testing, comparison, and analysis of several time series forecasting models. We use traditional regression baselines, tree-based models, and deep learning neural network structures.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in the system path to import src modules
sys.path.append(os.path.abspath("../src"))

# Setup style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

## 1. Load Data and Apply Feature Engineering Pipeline

In [ ]:
from config.paths import RAW_DATA_DIR
from data_pipeline.loader import load_dataset
from features.pipeline import FeatureEngineeringPipeline

raw_df = load_dataset(RAW_DATA_DIR / "train.csv")

# Since the full dataset has 913,000 rows, we sample the last 100,000 rows
# for faster training and demonstration in the notebook.
sampled_df = raw_df.sort_values("date").tail(100000).reset_index(drop=True)
print(f"Sampled dataset shape: {sampled_df.shape}")

# Apply features pipeline
pipeline = FeatureEngineeringPipeline()
features_df = pipeline.fit_transform(sampled_df)
print(f"Engineered dataset shape: {features_df.shape}")

In [ ]:
# Preview features
features_df.head()

## 2. Chronological Train-Test Split

In [ ]:
from models.split import train_test_split_time_series

X_train, X_test, y_train, y_test = train_test_split_time_series(features_df, test_size=0.20)

print(f"Train size: X_train = {X_train.shape}, y_train = {y_train.shape}")
print(f"Test size : X_test = {X_test.shape}, y_test = {y_test.shape}")

## 3. Run Benchmark Suite

We instantiate all registered forecasting models and train them using our `ModelTrainer` engine.

In [ ]:
from models.registry import get_models
from models.trainer import ModelTrainer

# Instantiate trainer
trainer = ModelTrainer()

# Get all default models (Baseline, Linear Regression, Random Forest, XGBoost, Prophet, LSTM)
models = get_models()
print(f"Training and evaluating models: {[m.__class__.__name__ for m in models]}")

# Run benchmark
results = trainer.benchmark(models, X_train, y_train, X_test, y_test)

In [ ]:
# Display benchmark rankings
results

## 4. Visual Comparison of Model Metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Compare RMSE
sns.barplot(data=results, x="Model", y="RMSE", ax=axes[0], palette="Blues_r")
axes[0].set_title("Model Comparison by RMSE (Lower is Better)")
axes[0].tick_params(axis='x', rotation=30)

# Compare R2 Score
sns.barplot(data=results, x="Model", y="R2", ax=axes[1], palette="Greens_d")
axes[1].set_title("Model Comparison by R2 (Higher is Better)")
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 5. Forecast Visualization

Let's select the best model and plot its predictions against actual values on the test set.

In [ ]:
best_model_name = results.iloc[0]["Model"]
best_model = trainer.models[best_model_name]
print(f"Best Performing Model: {best_model_name}")

# Generate predictions
preds = best_model.predict(X_test)

# Prepare comparison dataframe (taking latest 150 rows for clarity in visualization)
comparison_df = pd.DataFrame({
    "Actual": y_test,
    "Predicted": preds
}).tail(150).reset_index(drop=True)

plt.figure(figsize=(14, 6))
plt.plot(comparison_df["Actual"], label="Actual Sales", color="black", linewidth=1.5)
plt.plot(comparison_df["Predicted"], label=f"Forecast ({best_model_name})", color="dodgerblue", linestyle="--", linewidth=2)
plt.title(f"Model Predictions vs Actuals (Best Model: {best_model_name})")
plt.xlabel("Time (Steps)")
plt.ylabel("Sales")
plt.legend()
plt.show()

## 6. Model Explainability: Feature Importance

If the best performing model is a tree-based model (e.g. Random Forest, XGBoost), we can inspect feature importances.

In [ ]:
if hasattr(best_model, "feature_importances"):
    importances = best_model.feature_importances
    feat_imp = pd.DataFrame({
        "Feature": X_train.columns,
        "Importance": importances
    }).sort_values(by="Importance", ascending=False).reset_index(drop=True)
    
    plt.figure(figsize=(10, 8))
    sns.barplot(data=feat_imp.head(15), x="Importance", y="Feature", palette="viridis")
    plt.title(f"Top Feature Importances ({best_model_name})")
    plt.xlabel("Importance Score")
    plt.ylabel("Feature")
    plt.show()
else:
    print(f"The best model {best_model_name} does not expose feature importances directly.")